# Results (tabes and figures)

In [ ]:
import numpy as np
import pickle as pk
import pandas as pd

In [ ]:
%pip install openpyxl
%pip install --upgrade "kaleido>=1"

## Function to aggregate results

In [ ]:
def agrega_mat(aggr, mat):
    a = mat.copy()
    i,j = a.shape
    if type(aggr) == type(None):
        return a.reshape(i, j, -1)
    else:
        return np.concatenate((aggr.copy(), a.reshape(i,j,-1)), axis=2)

def corr_sp(A, B):
    return np.array([
            pd.Series(A[:,i]).corr(pd.Series(B[:,i]), method='spearman') 
            for i in range(A.shape[1])
    ])

def corr_global(A, B):
    return pd.Series(A.flatten()).corr(pd.Series(B.flatten()), method='spearman')


## Table general por muestreo

En esta tabla vamos a compara para 3 entradas diferentes (summer, winter y holyday) varias cosas:

1. La variablidad de las explicaciones para cada día, usando 5 muestras de entrenamiento aleatorio para un tamaño de muestra de 100, 500 y 1000. Vamos a comparas la variabilidad del modelo KernelSHAP con nuestra propuesta, bajo los mismos conjuntos.
2. La diferencia promedio entre los vectores de explicabilidad obtenidos con KernelSHAP y con nuestra propuesta por los 5 muestreos diferentes del conjunto de entrenamiento, para cada día y tamaño de muestra. Esto nos permite ver si hay diferencias de magnitud en los valores entre los dos métodos.
3. El porcentaje que cambia de signo los valores de explicabilidad de nuestro método con el método KernelSHAP, para cada día y tamaño de muestra. Esto nos permite ver si hay diferencias de explicación (positiva o negativa) entre los dos métodos. 
4. El coeficiente de correlación de Spearman entre los vectores de explicabilidad obtenidos con KernelSHAP y con nuestra propuesta, para cada día y tamaño de muestra. Esto nos permite ver si hay diferencias en el ordenamiento de importancia de las variables entre los dos métodos.
5. El tiempo de ejecución promedio de cada método para cada día y tamaño de muestra. Esto nos permite ver si hay diferencias en la eficiencia de los dos métodos.

El número de llamadas al modelo entrenado es constante para cáda método y es un dato que se va a poner en el caption de la tabla para no hacer una tabla con sólo dos valores diferentes.


In [ ]:
data_path = "./data/results/paper/"
table_ls = []

for test_sample in ["summer", "winter", "christmas"]:
    for train_size in [100, 500, 1000]:
        table_dict = {"X id": test_sample, "Train_size": train_size}
        hist_dfs = []        

        Xi, Phi = None, None
        error_Xi_Phi = []
        sign_change = []
        corr_spear, coeff_by_hour = [], []
        exec_xi, exec_phi = [], []

        for train_sample in range(1, 6):
            par_df = {}
            with open(data_path 
                        + 'raw/' 
                        + f'res_{test_sample}_{train_size}_{train_sample}.pkl', 
                      'rb') as f:
                res_KS = pk.load(f)
            with open(data_path 
                        + 'base/' 
                        + f'res_{test_sample}_{train_size}_{train_sample}.pkl', 
                      'rb') as f:
                res_CU = pk.load(f)

            Xi = agrega_mat(Xi, res_KS["matrix"])
            Phi = agrega_mat(Phi, res_CU["matrix"])
            error_Xi_Phi.append(
                np.mean(np.abs(res_KS["matrix"] - res_CU["matrix"])))
            sign_change.append(
                np.mean(np.sign(res_KS["matrix"]) != np.sign(res_CU["matrix"])))
            corr_spear.append(
                corr_global(res_KS["matrix"], res_CU["matrix"]))
            coeff_by_hour.append(
                corr_sp(res_KS["matrix"], res_CU["matrix"]))
            exec_xi.append(res_KS["exec_time"])
            exec_phi.append(res_CU["exec_time"])

            indicador = np.sign(res_KS["matrix"]) != np.sign(res_CU["matrix"])
            df_vals = pd.DataFrame({
                "explicability values": res_CU["matrix"].flatten(),
                "sign change": indicador.flatten()
            })         
            hist_dfs.append(df_vals)
        
        df_hist = pd.concat(hist_dfs) 
        df_hist.to_csv(
            data_path + 'tables/' + f"change_sign_{test_sample}_{train_size}.csv", 
            index=False) 

        table_dict["mean(std(xi_i^t))"] = [np.mean(np.std(Xi, axis=2))]
        table_dict["mean(std(phi_i^t))"] = [np.mean(np.std(Phi, axis=2))]
        table_dict["mean mean(abs(xi_i^t - phi_i^t))"] = [np.mean(error_Xi_Phi)]
        table_dict["std mean(abs(xi_i^t - phi_i^t))"] = [np.std(error_Xi_Phi)]
        table_dict["mean sign_change"] = [np.mean(sign_change)]
        table_dict["mean Spearman correlation coeficient"] = [np.mean(corr_spear)]
        table_dict["mean execution time Xi"] = [np.mean(exec_xi)]
        table_dict["std execution time Xi"] = [np.std(exec_xi)]
        table_dict["mean execution time Phi"] = [np.mean(exec_phi)]
        table_dict["std execution time Phi"] = [np.std(exec_phi)]
        table_ls.append(pd.DataFrame(table_dict))
table_df = pd.concat(table_ls)
table_df.to_csv(data_path + "tables/ks_vs_cu.csv", index=False)

## Formando la tabla para el artículo

In [ ]:
tabla = r"""
\begin{table}[htbp]
\centering
\caption{Robustness results with respect to sample size. Three different days from the test set were evaluated using 10 random sample sets per sample size.}
\label{tab:sample_size_results} 
\begin{adjustwidth}{-\extralength}{0cm}
\begin{center}
{\fontsize{7}{9}\selectfont 
\begin{tabular}{ """ + 9 * r">{\centering\arraybackslash}m{1.4cm}" + "}" + r"""
\hline
\textbf{$X$ ID} & \textbf{$X_{train}$ Sample size} & \textbf{$\text{std}(\xi_i^t)$} & \textbf{$\text{std}(\varphi_i^t)$} & \textbf{$\text{mean}(|\xi_i^t - \varphi_i^t|)$} & \textbf{Sign change (\%)} & \textbf{Mean Spearmen correlation coeficient}  & \textbf{Exec time $\xi$} & \textbf{Exec time $\psi$} \\ 
\hline
"""
for l in table_df.itertuples():
    tabla += (
        f"{l[1]} & {l[2]} & {round(l[3], 4)} & {round(l[4], 4)} & "
        + f"{round(l[5], 4)} $\pm$ {round(l[6], 4)} & "
        + f"{round(100*l[7], 2)}\% & {round(l[8], 4)} & "
        + f"{round(l[9], 2)} $\pm$ {round(l[10], 2)} & "
        + f"{round(l[11], 2)} $\pm$ {round(l[12], 2)} \\\\ \n"
    )
tabla += r"""\end{tabular}}
\end{center}
\end{adjustwidth}
\end{table}"""

print(tabla)

## Figura

Figura sobre la distribución de los valores de explicabilidad respecto a los valores que cambiaron de signo respecto a la metodología de KernelSHAP.

In [ ]:
import plotly.express as px

bar_df_l = []
b_l = []
for test_sample in ["summer", "winter", "christmas"]:
    a = pd.read_csv(data_path + 'tables/' + f"change_sign_{test_sample}_100.csv")
    a['X id'] = test_sample
    bar_df_l.append(a.copy())
    b = (a.groupby("sign change")
        .agg({"explicability values": 
            ["median", lambda x: x.quantile(0.75) - x.quantile(0.25)]})
        .reset_index()
        .round(4)
    )
    b_l.append(b)

bar_df = pd.concat(bar_df_l)

    
ply = px.box(
    bar_df, 
    x="explicability values", 
    color="sign change", 
    facet_col='X id',
    #title="Distribution of explainability values by sign change with respect to the sampled KernelSHAP method.",
    labels={"explicability values": "Explainability values", "sign change": "Sign Change"},
    color_discrete_map={
        True: '#5054af',
        False: '#afab50',
    },
    category_orders={"sign change": [False, True]}
)
ply.update_annotations(
    y=0.9,               
    font=dict(size=10, color='gray')
)
ply.add_annotation(
    text=f"Median: {b_l[0].iloc[0,1]}, IQR: {b_l[0].iloc[0,2]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.17, y=0.85,
    showarrow=False,
    font=dict(size=9, color='#afab50'),
)   
ply.add_annotation(
    text=f"Median: {b_l[0].iloc[1,1]}, IQR: {b_l[0].iloc[1,2]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.17, y=0.9,
    showarrow=False,
    font=dict(size=9, color='#5054af'),
)  
ply.add_annotation(
    text=f"Median: {b_l[1].iloc[0,1]}, IQR: {b_l[1].iloc[0,2]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.5, y=0.85,
    showarrow=False,
    font=dict(size=9, color='#afab50'),
)   
ply.add_annotation(
    text=f"Median: {b_l[1].iloc[1,1]}, IQR: {b_l[1].iloc[1,2]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.5, y=0.9,
    showarrow=False,
    font=dict(size=9, color='#5054af'),
)  
ply.add_annotation(
    text=f"Median: {b_l[2].iloc[0,1]}, IQR: {b_l[2].iloc[0,2]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.85, y=0.85,
    showarrow=False,
    font=dict(size=9, color='#afab50'),
)   
ply.add_annotation(
    text=f"Median: {b_l[2].iloc[1,1]}, IQR: {b_l[2].iloc[1,2]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.85, y=0.9,
    showarrow=False,
    font=dict(size=9, color='#5054af'),
)  

ply.update_xaxes(
    title_font=dict(size=10), 
    tickfont=dict(size=8),  
    ticklabelstandoff=-50  
)
ply.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",  # Quita el fondo exterior (lienzo)
    plot_bgcolor="rgba(0,0,0,0)",    # Quita el fondo interior (área del gráfico)
    legend=dict(
        orientation="h", yanchor="bottom", y= 1, xanchor="left", x=0,
        font=dict(size=10, color='gray')
    ),
)        
ply.show()
ply.write_image(data_path + 'tables/' + f"change_sign.pdf")
    

# Comparcion utilizando un esquema alternativo de CloseUp

Vamos a comparar respecto a la metodología KernelSHAP ambos esquemas de CloseUp. 

In [ ]:
data_path = "./data/results/paper/"
table_ls = []

for test_sample in ["summer", "winter", "christmas"]:
    table_dict = {"X id": test_sample}       

    error_Phi_Phi2 = []
    sign_change = []
    corr_spear = []
    exec_phi, exec_phi2 = [], []

    for train_sample in range(1, 6):
        par_df = {}
        with open(data_path + 'raw/' + f'res_{test_sample}_100_{train_sample}.pkl', 'rb') as f:
            res_KS = pk.load(f)
        with open(data_path + 'base/' + f'res_{test_sample}_100_{train_sample}.pkl', 'rb') as f:
            res_1 = pk.load(f)
        with open(data_path + 'wide/' + f'res_{test_sample}_100_{train_sample}.pkl', 'rb') as f:
            res_2 = pk.load(f)

        error_Phi_Phi2.append(
            np.mean(np.abs(res_1["matrix"] - res_2["matrix"])))
        sign_change.append(
            np.mean(np.sign(res_1["matrix"]) != np.sign(res_2["matrix"])))
        corr_spear.append(
            corr_global(res_1["matrix"], res_2["matrix"]))
        exec_phi.append(res_1["exec_time"])
        exec_phi2.append(res_2["exec_time"])

    table_dict["mean abs diff"] = [np.mean(error_Phi_Phi2).round(4)]
    table_dict["std abs diff"] = [np.std(error_Phi_Phi2).round(4)]
    table_dict["sign_change"] = [np.mean(sign_change).round(2)]
    table_dict["Spearman coef"] = [np.mean(corr_spear).round(4)]
    table_dict["mean execution time Phi"] = [np.mean(exec_phi).round(2)]
    table_dict["std execution time Phi"] = [np.std(exec_phi).round(2)]
    table_dict["mean execution time Phi2"] = [np.mean(exec_phi2).round(2)]
    table_dict["std execution time Phi2"] = [np.std(exec_phi2).round(2)]
    table_ls.append(pd.DataFrame(table_dict))
table_df = pd.concat(table_ls)
table_df.to_csv(data_path + "tables/base_vs_wide.csv", index=False)

# Generando la tabla de resultados

In [ ]:
tabla = r"""
\begin{table}[htbp]
\centering
\caption{Effect of alternative tree structures on the same interpretable components.}
\label{tab:alternative} 
\begin{center}
{\fontsize{7}{9}\selectfont 
\begin{tabular}{ """ + 6 * r">{\centering\arraybackslash}m{1.4cm}" + "}" + r"""
\toprule
\textbf{$X$ ID} & \textbf{$\text{mean}(|\varphi_i^t - \bar{\varphi}_i^t|)$} & \textbf{Sign change (\%)} & \textbf{Spearman coef}  & \textbf{Exec time $\phi$} & \textbf{Exec time $\bar{\phi$}} \\ 
\midrule
"""
for l in table_df.itertuples():
    tabla += (
        f"{l[1]} &  {l[2]} $\pm$ {l[3]} & {round(100*l[4])}\% & {l[5]} & "
        + f"{l[6]} $\pm$ {l[7]} & {l[8]} $\pm$ {l[9]} \\\\ \n"
    )
tabla += r"""\bottomrule 
\end{tabular}}
\end{center}
\end{table}"""

print(tabla)

## Comparación con usar o no la correccion de error

Aquí si, respecto a la metodología KernelSHAP, vamos a comparar el uso de la corrección de error en el método CloseUp.

In [ ]:
data_path = "./data/results/paper/"
table_ls = []

for test_sample in ["summer", "winter", "christmas"]:
    table_dict = {"X id": test_sample}       

    error1, error2 = [], []
    sign_change = []
    corr_spear = []

    for train_sample in range(1, 6):
        par_df = {}
        with open(data_path + 'raw/' + f'res_{test_sample}_100_{train_sample}.pkl', 'rb') as f:
            res_KS = pk.load(f)
        with open(data_path + 'base/' + f'res_{test_sample}_100_{train_sample}.pkl', 'rb') as f:
            res_1 = pk.load(f)
        with open(data_path + 'biased/' + f'res_{test_sample}_100_{train_sample}.pkl', 'rb') as f:
            res_2 = pk.load(f)

        error1.append(
            np.mean(np.abs(res_KS["matrix"] - res_1["matrix"])))
        error2.append(
            np.mean(np.abs(res_KS["matrix"] - res_2["matrix"])))
        sign_change.append(
            np.mean(np.sign(res_1["matrix"]) != np.sign(res_2["matrix"])))
        corr_spear.append(
            corr_global(res_1["matrix"], res_2["matrix"]))

    table_dict["mean abs"] = [np.mean(error1).round(4)]
    table_dict["std abs"] = [np.std(error1).round(4)]
    table_dict["mean abs biased"] = [np.mean(error2).round(4)]
    table_dict["std abs biased"] = [np.std(error2).round(4)]
    table_dict["sign_change"] = [np.mean(sign_change).round(2)]
    table_dict["Spearman coef"] = [np.mean(corr_spear).round(4)]
    table_ls.append(pd.DataFrame(table_dict))
table_df = pd.concat(table_ls)
table_df["Incremento (%)"] = (table_df["mean abs biased"] - table_df["mean abs"]) / table_df["mean abs"]
table_df.to_csv(data_path + "tables/base_vs_biased.csv", index=False)

# Tabla del efecto de corrección de error

In [ ]:
tabla = r"""
\begin{table}[htbp]
\centering
\caption{Effect of $\Psi$ in explainability values.}
\label{tab:psi_effect} 
\begin{center}
{\fontsize{7}{9}\selectfont 
\begin{tabular}{ """ + 4 * r">{\centering\arraybackslash}m{1.8cm}" + "}" + r"""
\toprule
\textbf{$X$ ID} & \textbf{error increment (\%)} & \textbf{Sign change (\%)} & \textbf{Spearman coef} \\ 
\midrule
"""
for l in table_df.itertuples():
    tabla += (
        f"{l[1]} & {round(100 * l[8],1)}\% & {round(100 * l[6])}\% & {l[7]} \\\\ \n"
    )
tabla += r"""\bottomrule 
\end{tabular}}
\end{center}
\end{table}"""

print(tabla)

## Effect of filter 

Sólo para el caso de *Summer* pero lo podríamos hacer para winter

In [ ]:
data_path = "./data/results/paper/"

diferences = []
sign_change = []
corr_spear = []

for train_sample in range(1, 6):
    par_df = {}
    with open(data_path + 'base/' + f'res_summer_100_{train_sample}.pkl', 'rb') as f:
        res_1 = pk.load(f)
    with open(data_path + 'filter/' + f'res_summer_100_{train_sample}.pkl', 'rb') as f:
        res_2 = pk.load(f)

    diferences.append(
        np.mean(np.abs(res_1["matrix"] - res_2["matrix"])))
    sign_change.append(
        np.mean(np.sign(res_1["matrix"]) != np.sign(res_2["matrix"])))
    corr_spear.append(
        corr_global(res_1["matrix"], res_2["matrix"]))

table_dict = {
    "mean abs diferences": [np.mean(diferences).round(4)],
    "std abs diferences": [np.std(diferences).round(4)],
    "sign_change": [np.mean(sign_change).round(2)],
    "Spearman coef": [np.mean(corr_spear).round(4)],
}
table_df = pd.DataFrame(table_dict)

table_df.to_csv(data_path + "tables/base_vs_filter.csv", index=False)

In [ ]:
table_df

# Tabla

Solo de referencia, no se publicará.

In [ ]:
tabla = r"""
\begin{table}[htbp]
\centering
\caption{Effect of filter in explainability values.}
\label{tab:filter_effect} 
\begin{center}
{\fontsize{7}{9}\selectfont 
\begin{tabular}{ """ + 3 * r">{\centering\arraybackslash}m{1.8cm}" + "}" + r"""
\toprule
\textbf{$X$ ID} & \textbf{Differences in $\xi_i^t} & \textbf{Sign change (\%)} & \textbf{Spearman coef} \\ 
\midrule
"""
for l in table_df.itertuples():
    tabla += (
        f"{l[1]} & {round(100 * l[3],1)}\% & {l[4]} \\\\ \n"
    )
tabla += r"""\bottomrule 
\end{tabular}}
\end{center}
\end{table}"""

print(tabla)

# Tabla sobre la repetibilidad en diferentes días del modelo de CloseUp con KernelShap

Para 50 días diferentes

In [ ]:
data_path = "./data/results/paper/"
table_ls = []

e1, e2, e4 = [], [], []
s1, s2, s4 = [], [], []
c1, c2, c4 = [], [], []
t1, t2, t4 = [], [], []

for test_sample in range(1, 51):
    par_df = {}
    with open(data_path + 'raw/' + f'res_{test_sample}.pkl', 'rb') as f:
        res_K = pk.load(f)
    with open(data_path + 'base/' + f'res_{test_sample}.pkl', 'rb') as f:
        res_1 = pk.load(f)
    with open(data_path + 'wide/' + f'res_{test_sample}.pkl', 'rb') as f:
        res_2 = pk.load(f)
    with open(data_path + 'tri/' + f'res_{test_sample}.pkl', 'rb') as f:
        res_4 = pk.load(f)

    e1.append(np.mean(np.abs(res_K["matrix"] - res_1["matrix"])))
    e2.append(np.mean(np.abs(res_K["matrix"] - res_2["matrix"])))
    e4.append(np.mean(np.abs(res_K["matrix"] - res_4["matrix"])))
    
    s1.append(np.mean(np.sign(res_K["matrix"]) != np.sign(res_1["matrix"])))
    s2.append(np.mean(np.sign(res_K["matrix"]) != np.sign(res_2["matrix"])))
    s4.append(np.mean(np.sign(res_K["matrix"]) != np.sign(res_4["matrix"])))
    
    c1.append(corr_global(res_K["matrix"], res_1["matrix"]))
    c2.append(corr_global(res_K["matrix"], res_2["matrix"]))
    c4.append(corr_global(res_K["matrix"], res_4["matrix"]))
    
    t1.append(res_1["exec_time"])
    t2.append(res_2["exec_time"])
    t4.append(res_4["exec_time"])

    indicador = np.sign(res_K["matrix"]) != np.sign(res_1["matrix"])
    df_1 = pd.DataFrame({
        "explicability values": res_1["matrix"].flatten(),
        "sign change": indicador.flatten()
    })
    df_1['Model'] = r"Delta"

    indicador = np.sign(res_K["matrix"]) != np.sign(res_2["matrix"])
    df_2 = pd.DataFrame({
        "explicability values": res_2["matrix"].flatten(),
        "sign change": indicador.flatten()
    })
    df_2['Model'] = r"Delta_altp"

    indicador = np.sign(res_K["matrix"]) != np.sign(res_4["matrix"])
    df_4 = pd.DataFrame({
        "explicability values": res_4["matrix"].flatten(),
        "sign change": indicador.flatten()
    })
    df_4['Model'] = r"Delta_alt"

    table_ls.append(df_1.copy())
    table_ls.append(df_4.copy())
    table_ls.append(df_2.copy())
    
table_dict = {
    "Model": [r'$\Delta$', r'$\Delta_{\sf alt}$', r'$\Delta_{\sf alt3}$'],
    "mean abs diferences": [np.mean(e1), np.mean(e2), np.mean(e4)],
    "std abs diferences": [np.std(e1), np.std(e2), np.std(e4)],
    "sign_change": [np.mean(s1), np.mean(s2), np.mean(s4)],
    "sign_change_std": [np.std(s1), np.std(s2), np.std(s4)],
    "Spearman coef": [np.mean(c1), np.mean(c2), np.mean(c4)],
    "Spearman std": [np.std(c1), np.std(c2), np.std(c4)],
    "Exec time": [np.mean(t1), np.mean(t2), np.mean(t4)],
    "Exec time std": [np.std(t1), np.std(t2), np.std(t4)],
}
table_df = pd.DataFrame(table_dict)

table_df.to_csv(data_path + "tables/base_alternative_50.csv", index=False)

table_plot = pd.concat(table_ls)

In [ ]:
res_1['model_calls'], res_4['model_calls'], res_2['model_calls']


In [ ]:
tabla = r"""
\begin{table}[htbp]
\centering
\caption{Effect of alternative tree structures on the same interpretable components using 50 randomly selected input vectors. }
\label{tab:alternative50} 
\begin{center}
{\fontsize{7}{9}\selectfont 
\begin{tabular}{ """ + 5 * r">{\centering\arraybackslash}m{1.8cm}" + "}" + r"""
\toprule
\textbf{Tree structure} & \textbf{$|\xi_i^t - \phi_i^t}|$} & \textbf{Sign change (\%)} & \textbf{Spearman coef} & Exec time\\ 
\midrule
"""
for l in table_df.itertuples():
    tabla += (
        f"{l[1]} & {round(l[2],4)} $\pm$ {round(l[3],4)} & {round(100 * l[4],1)}\% & {round(l[6],4)} $\pm$ {round(l[7],4)}& {round(l[8],4)} $\pm$ {round(l[9],4)} \\\\ \n"
    )
tabla += r"""\bottomrule 
\end{tabular}}
\end{center}
\end{table}"""

print(tabla)

In [ ]:
plot_info = (table_plot
             .groupby(['Model', 'sign change'])
             .agg({"explicability values": 
                    ["median", lambda x: x.quantile(0.75) - x.quantile(0.25)]})
             .round(4)             
            )
plot_info

In [ ]:
import plotly.express as px
from IPython.display import display, HTML

# 1. FORZAR LA CARGA DE MATHJAX EN EL ENTORNO (Crucial para VS Code/Jupyter)
display(HTML(
    '<script type="text/javascript" async src="https://cloudflare.com"></script>'
))

    
ply = px.box(
    table_plot, 
    x="explicability values", 
    color="sign change", 
    facet_col='Model',
    #title="Distribution of explainability values by sign change with respect to the sampled KernelSHAP method.",
    labels={"explicability values": "Explainability values", "sign change": "Sign Change"},
    color_discrete_map={
        True: '#5054af',
        False: '#afab50',
    },
    category_orders={"sign change": [False, True]}
)
custom_labels = {
    "Model=Delta": r"$\Delta$", 
    "Model=Delta_alt": r"$\Delta_{\sf alt}$",
    "Model=Delta_altp": r"$\Delta_{\sf alt}'$"}
ply.for_each_annotation(lambda a: a.update(text=custom_labels.get(a.text, a.text)))
ply.update_annotations(
    y=0.9,               
    font=dict(size=14, color='gray')
)
ply.add_annotation(
    text=f"Median: {plot_info.iloc[0,0]}, IQR: {plot_info.iloc[0,1]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.18, y=0.85,
    showarrow=False,
    font=dict(size=9, color='#afab50'),
)   
ply.add_annotation(
    text=f"Median: {plot_info.iloc[1,0]}, IQR: {plot_info.iloc[1,1]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.18, y=0.9,
    showarrow=False,
    font=dict(size=9, color='#5054af'),
)  
ply.add_annotation(
    text=f"Median: {plot_info.iloc[2,0]}, IQR: {plot_info.iloc[2,1]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.5, y=0.85,
    showarrow=False,
    font=dict(size=9, color='#afab50'),
)   
ply.add_annotation(
    text=f"Median: {plot_info.iloc[3,0]}, IQR: {plot_info.iloc[3,1]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.5, y=0.9,
    showarrow=False,
    font=dict(size=9, color='#5054af'),
)  
ply.add_annotation(
    text=f"Median: {plot_info.iloc[4,0]}, IQR: {plot_info.iloc[4,1]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.84, y=0.85,
    showarrow=False,
    font=dict(size=9, color='#afab50'),
)   
ply.add_annotation(
    text=f"Median: {plot_info.iloc[5,0]}, IQR: {plot_info.iloc[5,1]}",
    xref="paper", yref="paper",
    xanchor="center",
    x=.84, y=0.9,
    showarrow=False,
    font=dict(size=9, color='#5054af'),
)  

ply.update_xaxes(
    title_font=dict(size=10), 
    tickfont=dict(size=8),  
    ticklabelstandoff=-50  
)
ply.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",  # Quita el fondo exterior (lienzo)
    plot_bgcolor="rgba(0,0,0,0)",    # Quita el fondo interior (área del gráfico)
    legend=dict(
        orientation="h", yanchor="bottom", y= -0.1, xanchor="center", x=0.5,
        font=dict(size=9, color='gray')
    ),
)        
ply.show(include_mathjax='cdn')
ply.write_image(data_path + 'tables/' + f"change_sign.pdf")
    

In [ ]:
data_path = "./data/results/paper/"
table_ls = []

e1, e2 = [], []
s1, s2 = [], []
c1, c2 = [], []

for test_sample in range(1, 51):
    par_df = {}
    with open(data_path + 'raw/' + f'res_{test_sample}.pkl', 'rb') as f:
        res_K = pk.load(f)
    with open(data_path + 'base/' + f'res_{test_sample}.pkl', 'rb') as f:
        res_1 = pk.load(f)
    with open(data_path + 'biased/' + f'res_{test_sample}.pkl', 'rb') as f:
        res_2 = pk.load(f)

    e1.append(
        np.mean(np.abs(res_K["matrix"] - res_1["matrix"])))
    e2.append(
        np.mean(np.abs(res_K["matrix"] - res_2["matrix"])))
    s1.append(
        np.mean(np.sign(res_K["matrix"]) != np.sign(res_1["matrix"])))
    s2.append(
        np.mean(np.sign(res_K["matrix"]) != np.sign(res_2["matrix"])))
    c1.append(
        corr_global(res_K["matrix"], res_1["matrix"]))
    c2.append(
        corr_global(res_K["matrix"], res_2["matrix"]))
    
table_dict = {
    "Model": [r'Error correction', r'No error correction'],
    "mean abs diferences": [
        np.mean(e1), np.mean(e2)],
    "std abs diferences": [
        np.std(e1), np.std(e2)],
    "sign_change": [
        np.mean(s1), np.mean(s2)],
    "Spearman coef": [
        np.mean(c1), np.mean(c2)],
    "Spearman std": [
        np.std(c1), np.std(c2)],
}
table_df = pd.DataFrame(table_dict)

table_df.to_csv(data_path + "tables/error_correction_50.csv", index=False)

In [ ]:
table_df

In [ ]:
tabla = r"""
\begin{table}[htbp]
\centering
\caption{Correction error effect. }
\label{tab:correction50} 
\begin{center}
{\fontsize{7}{9}\selectfont 
\begin{tabular}{ """ + 4 * r">{\centering\arraybackslash}m{1.8cm}" + "}" + r"""
\toprule
\textbf{Tree structure} & \textbf{$|\xi_i^t - \phi_i^t|$} & \textbf{Sign change (\%)} & \textbf{Spearman coef} & Exec time\\ 
\midrule
"""
for l in table_df.itertuples():
    tabla += (
        f"{l[1]} & {round(l[2],4)} $\pm$ {round(l[3],4)} & {round(100 * l[4],1)}\% & {round(l[5],4)} $\pm$ {round(l[6],4)} \\\\ \n"
    )
tabla += r"""\bottomrule 
\end{tabular}}
\end{center}
\end{table}"""

print(tabla)

In [ ]:
data_path = "./data/results/paper/"

Phi = None
for test_sample in range(1, 51):
    with open(data_path + 'base/' + f'res_{test_sample}.pkl', 'rb') as f:
        res = pk.load(f)
    Phi = agrega_mat(Phi, res["matrix"])

varabilidad = Phi.std(axis=2)

fig = px.imshow(
    varabilidad,
    labels=dict(x="hour", y="Interpretable component", color=r"$\text{std}(\varphi_i^t)$"),
                x=[str(i + 1) for i in range(36)],
                y=res['comp_names'],
                aspect="auto",
                color_continuous_scale='Viridis', origin='lower' #RdBu_r
               )
fig.update_xaxes(tickangle=0, tickfont=dict(size=10))
fig.update_yaxes(tickangle=0, tickfont=dict(size=10))
fig.update_layout(
    bargap=0.4,
    annotations=[
        dict(
            text=r"$\text{std}(\varphi_i^t)$", 
            xref="paper", yref="paper", 
            x=1.17, y=1.05,                     
            showarrow=False, textangle=0, font=dict(size=18)
        )
    ] 
)
fig.update_coloraxes(colorbar=dict(title=None))
fig.show()
fig.write_image(data_path + 'tables/' + f"var_heatmap.pdf")


In [ ]:
data_path = "./data/results/paper/"

with open(data_path + 'base/res_winter_100_1.pkl', 'rb') as f:
    res_w = pk.load(f)
with open(data_path + 'base/res_summer_100_1.pkl', 'rb') as f:
    res_s = pk.load(f)
with open(data_path + 'base/res_christmas_100_1.pkl', 'rb') as f:
    res_c = pk.load(f)

idx = [3, 2, 1, 0, 5, 4, 6, 7, 8, 9]

leyendas = [res_s['comp_names'][idx[i]].strip().replace('before','').ljust(23) for i in range(10)]

def genera_df(A, ID, lista):
    B = A / A.sum(axis=0)
    df = pd.DataFrame({'Explainability vectors': B.T.flatten()})   
    df['Time step (h)'] = df.index // len(lista)
    dfs['Interpretable component'] = [lista[i % len(lista)] for i in df.index]
    dfs['X']=ID


dfs = pd.DataFrame({'Explainability vectors': (np.abs(res_s['matrix'][idx,:]) / np.abs(res_s['matrix'][idx,:]).sum(axis=0)).T.flatten()})   
dfs['Time step (h)'] = dfs.index // len(leyendas)
dfs['Interpretable component'] = [leyendas[i % len(leyendas)] for i in dfs.index]
dfs['X']="Summer"

dfw = pd.DataFrame({'Explainability vectors': (np.abs(res_w['matrix'][idx,:]) / np.abs(res_w['matrix'][idx,:]).sum(axis=0)).T.flatten()})   
dfw['Time step (h)'] = dfw.index // len(leyendas)
dfw['Interpretable component'] = [leyendas[i % len(leyendas)] for i in dfw.index]
dfw['X']="Winter"

dfc = pd.DataFrame({'Explainability vectors': (np.abs(res_c['matrix'][idx,:]) / np.abs(res_c['matrix'][idx,:]).sum(axis=0)).T.flatten()})   
dfc['Time step (h)'] = dfc.index // len(leyendas)
dfc['Interpretable component'] = [leyendas[i % len(leyendas)] for i in dfc.index]
dfc['X']="Holiday"

df_fig = pd.concat([dfs, dfw, dfc])


In [ ]:
import plotly.express as px

# 1. Definimos el orden estricto de tus componentes interpretables
orden_componentes = [
    'load_demand_72-180',
    'load_demand_24-72',
    'load_demand_12-24',
    'load_demand_1-12',
    'calendar_24-180',
    'calendar_day_before',
    'calendar_future',
    'weather_before',
    'weather_future',
    'holiday'
]

# 2. Paleta Corporativa Refinada en ese mismo orden exacto
colores_profesionales = [
    # --- LOAD DEMAND (Azules) ---
    "#1E293B",  # load_demand_72-180
    "#2F456A",  # load_demand_24-72
    "#3B82F6",  # load_demand_12-24
    "#93C5FD",  # load_demand_1-12

    # --- CALENDAR (Verdes) ---
    "#14532D",  # calendar_24-180
    "#0F766E",  # calendar_day_before
    "#10B981",  # calendar_future

    # --- CLIMA (Magenta / Morado) ---
    "#701A75",  # weather_before
    "#A855F7",  # weather_future

    # --- EVENTOS ESPECIALES (Amarillo Champagne) ---
    "#D97706"   # holiday
]

# Generamos el gráfico de líneas con relleno en subplots
fig = px.area(
    df_fig, 
    x='Time step (h)', 
    y='Explainability vectors', 
    color='Interpretable component',
    facet_col='X',
    title="Gráfica de Líneas con Relleno por Hora (36 Horas)",
    color_discrete_sequence=colores_profesionales,
    # OBLIGATORIO: Forzamos a Plotly a ordenar los datos según nuestra lista
    category_orders={'Interpretable component': orden_componentes} 
)

# 3. Eliminamos bordes y aplicamos el truco de inyección directa de color puro
fig.update_traces(
    line=dict(width=0),               # Quita la línea oscura del borde
    marker=dict(size=0)               # Esconde marcadores puntuales
)

# Forzamos a la propiedad 'fillcolor' a usar el HEX corporativo sin degradarse
for trace in fig.data:
    if trace.line.color:
        trace.fillcolor = trace.line.color

# 4. LIMPIEZA DE TÍTULOS DE SUBPLOTS (Remueve el "X=" automático de Plotly)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# 5. Configuración del diseño premium con leyenda vertical a la derecha
fig.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",    # Fondo exterior transparente
    plot_bgcolor="rgba(0,0,0,0)",     # Fondo interior transparente

    legend=dict(
        orientation="v",              # Orientación vertical
        yanchor="middle", 
        xanchor="left",
        y=0.5,                        # Centrado verticalmente
        x=1.02,                       # Posición limpia a la derecha del gráfico
        font=dict(size=11)
    )
)

# Cuadrícula sutil de fondo para un acabado de publicación científica o reporte corporativo
fig.update_yaxes(showgrid=True, gridcolor='rgba(210, 210, 210, 0.15)')
fig.update_xaxes(showgrid=False)

fig.show()
fig.write_image(data_path + 'tables/' + f"fig_lines_area_professional.pdf")
